# Multi-Task GRL — Tox21 Training Notebook
**Compatible with Kaggle (GPU P100/T4) and Google Colab (GPU T4).**

Steps:
1. Check GPU status
2. Install PyTorch Geometric and other dependencies
3. Clone the repository
4. W&B logging login (uses secrets if available)
5. Load configuration
6. Download & process Tox21 dataset
7. Build DataLoaders
8. Build multi-task model (GINEConv backbone)
9. Train model (PCGrad + Uncertainty Weighting)
10. Quick inference sanity check (e.g. Aspirin)
11. Save model checkpoint and metrics JSON
12. Plot training and loss curves

In [ ]:
# ── Cell 1: Check GPU ─────────────────────────────────────────────────────
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout if result.returncode == 0 else 'No GPU detected! Please enable GPU in settings (Accelerator → GPU P100/T4 on Kaggle, or Runtime → Change runtime type → T4 GPU on Colab).')

In [ ]:
# ── Cell 2: Install dependencies ──────────────────────────────────────────
import torch
print(f'PyTorch version: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')

TORCH_VER = torch.__version__.split('+')[0]
CUDA_VER  = 'cu121' if torch.cuda.is_available() else 'cpu'

!pip install -q torch-scatter torch-sparse -f https://data.pyg.org/whl/torch-{TORCH_VER}+{CUDA_VER}.html
!pip install -q torch-geometric
!pip install -q rdkit-pypi wandb scikit-learn ogb tqdm pyyaml matplotlib
print('All dependencies installed!')

In [ ]:
# ── Cell 3: Detect environment and clone repo ────────────────────────────
import os
import sys

IS_COLAB = 'google.colab' in sys.modules or os.path.exists('/content')
BASE_DIR = '/content' if IS_COLAB else '/kaggle/working'
REPO_DIR = os.path.join(BASE_DIR, 'graphmol')

REPO_URL = 'https://github.com/dreamybear66/Multi-Task-Graph-Representation-Learning-for-Molecular-Property-Prediction.git'

if not os.path.exists(REPO_DIR):
    print(f'Cloning repository into {REPO_DIR}...')
    !git clone {REPO_URL} {REPO_DIR}
else:
    print(f'Updating repository in {REPO_DIR}...')
    !cd {REPO_DIR} && git pull

%cd {REPO_DIR}
!ls -la

In [ ]:
# ── Cell 4: W&B Login (Environment/Secrets Aware) ─────────────────────────
import wandb

USE_WANDB = False
api_key = os.environ.get('WANDB_API_KEY')

# 1. Try Kaggle secrets
if not api_key:
    try:
        from kaggle_secrets import UserSecretsClient
        secrets = UserSecretsClient()
        api_key = secrets.get_secret('WANDB_API_KEY')
        print('Loaded API key from Kaggle Secrets.')
    except Exception:
        pass

# 2. Try Colab secrets
if not api_key:
    try:
        from google.colab import userdata
        api_key = userdata.get('WANDB_API_KEY')
        print('Loaded API key from Colab Secrets.')
    except Exception:
        pass

if api_key:
    try:
        wandb.login(key=api_key)
        print('W&B login successful!')
        USE_WANDB = True
    except Exception as e:
        print(f'W&B login failed: {e}')
else:
    print('W&B key not found in Secrets. If you wish to use W&B logging, please add WANDB_API_KEY to your Kaggle Secrets or Colab Secrets. Proceeding without W&B logging.')

In [ ]:
# ── Cell 5: Load config ───────────────────────────────────────────────────
import yaml

sys.path.insert(0, REPO_DIR)

with open('configs/tox21_gin5.yaml') as f:
    cfg = yaml.safe_load(f)

# Override configurations dynamically for current runtime environment
cfg['use_wandb'] = USE_WANDB
cfg['data_dir']  = os.path.join(REPO_DIR, 'data')
cfg['checkpoint_dir'] = os.path.join(BASE_DIR, 'checkpoints/v1.0.0')

print('Config loaded:')
for k, v in cfg.items():
    print(f'  {k}: {v}')

In [ ]:
# ── Cell 6: Download and process Tox21 dataset ───────────────────────────
from src.data.dataset import Tox21Dataset

data_dir = cfg['data_dir']
os.makedirs(data_dir, exist_ok=True)

print('Loading Tox21 (downloads + featurizes on first run)…')
train_dataset = Tox21Dataset(root=data_dir, split='train')
val_dataset   = Tox21Dataset(root=data_dir, split='val')
test_dataset  = Tox21Dataset(root=data_dir, split='test')

print(f'Train: {len(train_dataset)} | Val: {len(val_dataset)} | Test: {len(test_dataset)}')
print(f'Sample x shape: {train_dataset[0].x.shape}')    # [num_atoms, 85]
print(f'Sample y shape: {train_dataset[0].y.shape}')    # [12]

In [ ]:
# ── Cell 7: Build DataLoaders ─────────────────────────────────────────────
from torch_geometric.loader import DataLoader

BS = cfg['batch_size']
train_loader = DataLoader(train_dataset, batch_size=BS, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=BS, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=BS, shuffle=False, num_workers=2, pin_memory=True)

print(f'Batches — Train: {len(train_loader)} | Val: {len(val_loader)} | Test: {len(test_loader)}')

In [ ]:
# ── Cell 8: Build model ───────────────────────────────────────────────────
from src.models.model import MTGRLModel

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')

model = MTGRLModel.from_config(cfg)
model.count_parameters()

In [ ]:
# ── Cell 9: Train (Full Model — PCGrad + Uncertainty) ────────────────────
from src.training.trainer import Trainer
import random, numpy as np

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(cfg.get('seed', 42))

trainer = Trainer(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    test_loader=test_loader,
    cfg=cfg,
    device=device,
)

test_metrics = trainer.run()

print('\n===== FINAL TEST RESULTS =====')
for task, auc in sorted(test_metrics.items()):
    print(f'  {task:<25} {auc:.4f}')

In [ ]:
# ── Cell 10: Quick sanity check — predict Aspirin ─────────────────────────
model.load_state_dict(torch.load(f"{cfg['checkpoint_dir']}/model.pt", map_location=device))
model.eval()

aspirin = 'CC(=O)Oc1ccccc1C(=O)O'
preds = model.predict(aspirin, device=device)

print(f'Predictions for Aspirin ({aspirin}):')
for task, prob in preds.items():
    bar = '█' * int(prob * 20)
    flag = ' ⚠️' if prob >= 0.5 else ''
    print(f'  {task:<20} {prob:.4f}  {bar}{flag}')

In [ ]:
# ── Cell 11: Save metrics JSON ────────────────────────────────────────────
import json

metrics_path = f"{cfg['checkpoint_dir']}/test_metrics.json"
with open(metrics_path, 'w') as f:
    json.dump(test_metrics, f, indent=2)

print(f'Metrics saved to {metrics_path}')
print(f'Checkpoint: {cfg["checkpoint_dir"]}/model.pt')
print('\n>>> Download model.pt from the Output files! <<<')

In [ ]:
# ── Cell 12: Plot loss curves ─────────────────────────────────────────────
import matplotlib.pyplot as plt

history = trainer.history
if history:
    epochs      = [r['epoch'] for r in history]
    train_loss  = [r['train_loss'] for r in history]
    val_auc     = [r['val_avg_auc'] for r in history]
    conflict    = [r['conflict_rate'] for r in history]

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    axes[0].plot(epochs, train_loss, color='#46f1d3'); axes[0].set_title('Train Loss')
    axes[1].plot(epochs, val_auc, color='#c6c0ff');   axes[1].set_title('Val Avg AUC')
    axes[2].plot(epochs, conflict, color='#f87171');  axes[2].set_title('Conflict Rate')

    for ax in axes:
        ax.set_xlabel('Epoch')
        ax.grid(alpha=0.3)

    plt.tight_layout()
    os.makedirs(cfg['checkpoint_dir'], exist_ok=True)
    plt.savefig(f"{cfg['checkpoint_dir']}/training_curves.png", dpi=150)
    plt.show()
    print('Plot saved!')
else:
    print('No training history found to plot.')